In [1]:
import pandas as pd
#import modin.pandas as pd
#import ray
import numpy as np
from matplotlib import pyplot as plt
import seaborn as sns
sns.set(style='whitegrid',font_scale=2)
import os
import pickle
import time
os.chdir(os.getcwd())


In [4]:
os.getcwd()

'/Users/daniel.garger/Desktop/Projects/C-PATH/TTP prediction/src'

In [2]:
#ae=pd.read_csv('../data/ttp_ae_standardised_with_time.csv',low_memory=False,index_col=0)
#ce=pd.read_csv('../data/ttp_ce_standardised_with_time.csv',low_memory=False,index_col=0)
#cm=pd.read_csv('../data/ttp_cm_standardised_with_time.csv',low_memory=False,index_col=0)
ce=pd.read_csv('../../C-Path_data/preprocessing/ce_cleaned_up_terms.csv',low_memory=False,index_col=0)
cm=pd.read_csv('../../C-Path_data/preprocessing/cm_cleaned_up_terms.csv',low_memory=False,index_col=0)
ae=pd.read_csv('../../C-Path_data/preprocessing/ae_cleaned_up_terms.csv',low_memory=False,index_col=0)
mh=pd.read_csv('../../C-Path_data/preprocessing/mh_cleaned_up_terms.csv',low_memory=False,index_col=0)
#ce_mh=pd.read_csv('../data/ttp_ce_mh.csv',low_memory=False,index_col=0)
#cm_mh=pd.read_csv('../data/ttp_cm_mh.csv',low_memory=False,index_col=0)
#mh=pd.read_csv('../data/ttp_mh.csv',low_memory=False,index_col=0)


In [3]:
all=pd.concat([ae,ce,cm,mh],ignore_index=True)

In [4]:
all_std_terms=all.loc[:,['USUBJID','STD_AETERM','STD_CETERM','STD_CMINDC','STD_MHTERM']]

In [30]:
all_std_terms.index

RangeIndex(start=0, stop=293793, step=1)

In [5]:
#all_std_terms.sum(axis=1)
all_terms=all[~all['STD_AETERM'].isna()]['STD_AETERM'].tolist() + \
                    all[~all['STD_CETERM'].isna()]['STD_CETERM'].tolist() +\
                    all[~all['STD_CMINDC'].isna()]['STD_CMINDC'].tolist() +\
                    all[~all['STD_MHTERM'].isna()]['STD_MHTERM'].tolist()
all['ALL_STD_TERMS']=all_terms


In [6]:
terms=all['ALL_STD_TERMS'].value_counts()

In [8]:
with open("../data/all_medical_terms.txt", "w") as output:
    output.write(str(terms.index.tolist()))


In [46]:
yes=[r'CHEST PAIN']
yes2=[r'PLEU']
no=[r"q"]
all[(all['ALL_STD_TERMS'].str.contains('|'.join(yes),na=False))&(all['ALL_STD_TERMS'].str.contains('|'.join(yes2),na=False))&
                            (~all['ALL_STD_TERMS'].str.contains('|'.join(no),na=False))]['ALL_STD_TERMS'].value_counts()#.index.tolist()

PLEURITIC CHEST PAIN                                                                                                                                          49
PLEURITIC CHEST PAIN DUE TO TB                                                                                                                                22
PLEURITIC CHEST PAIN;                                                                                                                                          4
PLEURITIC CHEST PAIN DUE TO TB.                                                                                                                                3
PLEURITIC CHEST PAIN (DUE TO TB)                                                                                                                               2
COSTOCHONDRITIS AND PLEURITIC CHEST PAIN                                                                                                                       2
PLEURITIC CHEST PAIN DUE TO CHEST 

## From chatGPT

In [ ]:
abbreviations = {
    'Fracture Hip': 'Fracture of the hip',
    'SLE': 'Systemic lupus erythematosus',
    'C/O Poor Focus LT. E': 'Complaining of poor focus in the left eye',
    'RT Eye Blindness': 'Right eye blindness',
    'Embassade Vis Fr Far': 'Embarrassed vision from far',
    'Vision Impairment': 'Reduction in the clarity or sharpness of vision',
    'Blurring': 'Blurred vision',
    'Down Vision OS': 'Vision is decreased in the left eye',
    'Difficult Vision': 'Difficulty in seeing clearly',
    'Mancular Degeneratio': 'Macular degeneration',
    'Dist Radial Fracture': 'Distal radial fracture',
    'Hosp for Tx of INH R TB': 'Hospitalization for treatment of INH-resistant TB',
    'Black Spots Occasion': 'Black spots occasionally seen in vision',
    'Shoulder Pain, Trtd Chiropract': 'Shoulder pain treated by chiropractic',
    'Pt Expired': 'Patient expired',
    'Tempor, Ileostomy': 'Temporary ileostomy',
    'Frostbite Feet': 'Freezing of feet',
    'Perirectal Pain': 'Pain around the rectum',
    'Cold Productive Cough': 'Cold with productive cough',
    'DKA': 'Diabetic ketoacidosis',
    'Penumocytis Pneumia': 'Pneumocystis pneumonia',
    'Hospilized-Cancer': 'Hospitalized for cancer',
    'Hosp for Pnumonia': 'Hospitalization for pneumonia',
    'Ant Eubcapsular Cata': 'Anterior capsular cataract',
    'Hpgly Muscpain Bronc': 'Hypertrophied muscle pain bronchitis',
    'Thrombosis ?': 'Possible thrombosis',
    'Ankle Sprain vs Frac': 'Ankle sprain versus fracture',
    'Attemp Suicide (GSW)': 'Attempted suicide by gunshot wound',
    'Ca Diagnosis': 'Cancer diagnosis',
    'Enlarged Prostate': 'Benign prostatic hyperplasia',
    'Endoscopic Exam-Live': 'Endoscopic examination of liver',
    'Status Asthmaticus': 'Severe asthma attack',
    'Amoxicillin 500 mg': 'Amoxicillin 500 milligrams',
    'Temp Elevated 103 Dental Crisi': 'Temperature elevated to 103 during dental crisis',
    'Alcohol Withdraw': 'Alcohol withdrawal',
    'Unmanaged Hypertension': 'Hypertension not well-controlled',
    'Ostheolytic Lesion, (see Form)': 'Osteolytic lesion (see form)',
    'Injuries Auto Accide': 'Injuries from an automobile accident',
    'Potts Disease': 'Spinal tuberculosis',
    'Post-Cortal Dysuria': 'Dysuria following corticosteroid therapy',
    'Vision Contrast': 'Contrast sensitivity of vision',
    'Inc Total Bilirubin': 'Increased total bilirubin',
    'Gastric Reflux': 'Gastroesophageal reflux',
    'Thigh Nuscle Pain': 'Thigh muscle pain',
    'Epigastric Pain': 'Pain in the upper abdomen',
    'Nocturnal Dyspocea': 'Difficulty breathing at,

    
    dd={
    'FRACTURE HIP': 'Fracture of the hip',
    'SLE': 'Systemic lupus erythematosus',
    'SLIGHT BLURRINESS': 'Slight blurriness of vision',
    'C/O POOR FOCUS LT. E': 'Complaining of poor focus in left eye',
    'RT EYE BLINDNESS': 'Blindness in right eye',
    'EMBASSADE VIS FR FAR': 'Embarrassed by vision from far distance',
    'VISION IMPAIRMENT': 'Impairment of vision',
    'BLURRING': 'Blurring of vision',
    'DOWN VISION OS': 'Downward vision in the left eye',
    'DIFFICULT VISION': 'Difficulty with vision',
    'MANCULAR DEGENERATIO': 'Macular degeneration',
    'DIST RADIAL FRACTURE': 'Distal radial fracture',
    'HOSP FOR TX OF INH R TB': 'Hospitalization for treatment of INH-resistant tuberculosis',
    'BLACK SPOTS OCCASION': 'Occasional black spots in vision',
    'SHOULDER PAIN, TRTD CHIROPRACT': 'Shoulder pain treated by chiropractic therapy',
    'PT EXPIRED': 'Patient expired',
    'TEMPOR, ILEOSTOMY': 'Temporary ileostomy',
    'FROSTBITE FEET': 'Frostbite of the feet',
    'PERIRECTAL PAIN': 'Pain in the area around the rectum',
    'COLD PRODUCTIVE COUG': 'Cold with productive cough',
    'DKA': 'Diabetic ketoacidosis',
    'PENUMOCYPTIS PNEUMIA': 'Pneumococcal pneumonia',
    'HOSPILIZED-CANCER': 'Hospitalized for cancer',
    'HOSP FOR PNUMONIA': 'Hospitalization for pneumonia',
    'ANT EUBCAPSULAR CATA': 'Anterior capsular cataract',
    'HPGLY MUSCPAIN BRONC': 'Hyperglycemic muscle pain with bronchitis',
    'THROMBOSIS ?': 'Thrombosis, uncertain diagnosis',
    'ANKLE SPRAIN VS FRAC': 'Ankle sprain versus fracture',
    'ATTEMP SUICIDE (GSW)': 'Attempted suicide by gunshot wound',
    'CA DIAGNOSIS': 'Cancer diagnosis',
    'ENLARGED PROSTATE': 'Enlarged prostate',
    'ENDOSCOPIC EXAM-LIVE': 'Endoscopic examination of the liver',
    'STATUS ASTHMATIANS': 'Status asthmaticus',
    'AMOXICILLIN 500 MG': 'Amoxicillin 500 mg',
    'TEMP ELEVATED 103 DENTAL CRISI': 'Temporary elevation of temperature to 103 degrees during dental crisis',
    'ALCOHOL WITHDRAW': 'Alcohol withdrawal',
    'UNMANAGED HYPERTENSION': 'Uncontrolled hypertension',
    'OSTHEOLYTIC LESION, (SEE FORM)': 'Osteolytic lesion (see form)',
    'INJURIES AUTO ACCIDE': 'Injuries from automobile accident',
    'POTTS DISEASE': 'Pott disease',
    'POST-CORTAL DYSURIA': 'Dysuria after corticosteroid use',
    'VISION CONTRAST': 'Contrast sensitivity of vision',
    'INC TOTAL BILIRUBIN': 'Increased

